In [1]:
import pandas as pd
import yaml
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from pathlib import Path

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn

In [3]:
model_version = '1_0_5'
chunk_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{model_version}_standard_global__20260224/'
parquet_name = f'veg_model_zonal_stats_v{model_version}_20260224_19_21_07.parquet'

In [4]:
%%time

# Reads gross outputs parquet table
df = pd.read_parquet(f'{chunk_stats_folder}{parquet_name}')
df["WDPA_high_protection"] = df["WDPA"].isin([1, 2, 3]).astype(int)
df["country_name"] = df["country_name"].replace({
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Russian Federation": "Russia",
    "Democratic Republic of the Congo": "DR Congo",
    "United States of America (the)": "USA"
})

print(f"Rows in df: {len(df)}")
# df

Rows in df: 27999047
CPU times: user 15.6 s, sys: 9.46 s, total: 25 s
Wall time: 10.7 s


Create simplified/abbreviated table that can be used in PyGWalker

In [5]:
%%time

# To create a wide-format table (with fluxes only, not areas of flux densities).
# Drops a few contextual columns to reduce the number of rows
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

id_cols_by_land_state = [
    # "adm0",
    # "country_name",
    "region",
    "land_state_node",
    "land_state_meaning",
    "land_state_broad_class",
    "land_state_detailed_class",
    "WDPA",
    "WDPA_type",
    "cont_eco",
    "continent",
    "continent_ecozone",
    "Landmark",
    "starting_composite_primary_forest",
    "year",
    # "tile_id",
    # "area_ha",
    # "density__Mg_ha",
]

df_wide_by_land_state = (
    df.groupby(id_cols_by_land_state + ["analysis_layer"], dropna=False)["value"]
      .sum()
      .unstack("analysis_layer")
      .reset_index()
)

df_wide_by_land_state

CPU times: user 15.5 s, sys: 3.52 s, total: 19 s
Wall time: 18.7 s


analysis_layer,region,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,WDPA,WDPA_type,cont_eco,continent,continent_ecozone,...,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2
0,Europe,21200000,Gain of non-oil palm planted trees,tree,tree_gain,0,NA,0,NaN,NaN,...,-0.055070,-0.266904,NaN,NaN,-0.211834,-0.055070,-0.266904,-0.266904,NaN,NaN
1,Europe,21200000,Gain of non-oil palm planted trees,tree,tree_gain,0,NA,0,NaN,NaN,...,-0.552837,-2.679406,NaN,NaN,-2.126570,-0.552837,-2.679406,-2.679406,NaN,NaN
2,Europe,21200000,Gain of non-oil palm planted trees,tree,tree_gain,0,NA,0,NaN,NaN,...,-0.550710,-2.669098,NaN,NaN,-2.118388,-0.550710,-2.669097,-2.669097,NaN,NaN
3,Europe,21200000,Gain of non-oil palm planted trees,tree,tree_gain,0,NA,0,NaN,NaN,...,-0.418023,-2.026012,NaN,NaN,-1.607989,-0.418023,-2.026012,-2.026012,NaN,NaN
4,Europe,21200000,Gain of non-oil palm planted trees,tree,tree_gain,0,NA,0,NaN,NaN,...,-0.265035,-1.284533,NaN,NaN,-1.019498,-0.265035,-1.284533,-1.284533,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739034,NaN,63900000,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,11,Not Reported,7022,Europe,Water,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4427.398438,NaN,NaN
739035,NaN,63900000,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,11,Not Reported,7022,Europe,Water,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2602.613525,NaN,NaN
739036,NaN,63900000,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,11,Not Reported,7022,Europe,Water,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2692.835938,NaN,NaN
739037,NaN,70000000,Not in decision tree,no_flux,no_flux,0,NA,4008,Asia,Subtropical humid forest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
%%time
# Number of options for each contextual column

summary = {
    col: df[col].nunique()
    for col in df.columns
    if col not in ["index", "value", "analysis_layer", "area_ha", "density__Mg_ha"]
}

pd.Series(summary).sort_values(ascending=False)

In [6]:
%%time

layers_to_drop = [
    "carbon_density__non_soil__MgC_ha",
    cn.agc_gross_emis_pattern,
    cn.bgc_gross_emis_pattern,
    cn.deadwood_c_gross_emis_pattern,
    cn.litter_c_gross_emis_pattern,
    cn.net_flux_all_C_pools_CO2_only_pattern,
    cn.agc_gross_removals_pattern,
    cn.bgc_gross_removals_pattern,
    cn.deadwood_c_gross_removals_pattern,
    cn.litter_c_gross_removals_pattern,
    cn.net_flux_agc_pattern,
    cn.net_flux_bgc_pattern,
    cn.net_flux_deadwood_c_pattern,
    cn.net_flux_litter_c_pattern,
    cn.ch4_gross_emis_pattern,
    cn.n2o_gross_emis_pattern
]

df_outputs_dropped = df[~df["analysis_layer"].isin(layers_to_drop)].reset_index(drop=True)
# df_outputs_dropped

CPU times: user 1.9 s, sys: 1.21 s, total: 3.1 s
Wall time: 3.16 s


In [7]:
%%time
# Drops contextual layers sequentially to get a sense of how many combination rows are added by including each one,
# i.e. how many rows are lost when I drop that one column, given that all the other columns are still present.
# per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

# Contextual layers that are essentially text versions of others, so they need to be dropped in order to assess how much complexity each contextual layer gives 
# (since they are redundant in terms of complexity)
df_by_context = df_outputs_dropped.drop(columns=['country_name', 'land_state_meaning', 'continent_ecozone', 'WDPA_type', 'region', 'continent', 'land_state_detailed_class', 'land_state_broad_class'])

base = len(df_by_context)
print(f"Rows in gross_emissions__all_C_pools__CO2_only__MgCO2 with all contextual layers: {base}")

cols_to_check = ['tile_id', 'adm0', 'cont_eco', 'WDPA', 'year', 'Landmark', 'starting_composite_primary_forest', 'WDPA_high_protection']

for col in cols_to_check:

    # Drops specified contextual layers to reduce the number of rows in the table (although dropping adm0 or land_state seems to remove only a few 10s of thousands of rows)
    cols_to_sum = ["value", "area_ha", "density__Mg_ha"]
    
    group_cols = [
        c for c in df_by_context.columns
        if c not in [col] + cols_to_sum  # Contextual columns to drop
    ]
    
    df_agg = (
        df_by_context.groupby(group_cols, dropna=False)
          .size()
          .reset_index()
    )
    reduced=len(df_agg)

    print(f"{col:35s} removing it reduces rows by {base - reduced}")

Rows in gross_emissions__all_C_pools__CO2_only__MgCO2 with all contextual layers: 8178188
tile_id                             removing it reduces rows by 3525055
adm0                                removing it reduces rows by 2530578
cont_eco                            removing it reduces rows by 3489651
WDPA                                removing it reduces rows by 3595859
year                                removing it reduces rows by 6778870
Landmark                            removing it reduces rows by 1399031
starting_composite_primary_forest   removing it reduces rows by 2331360
WDPA_high_protection                removing it reduces rows by 0
CPU times: user 21.6 s, sys: 3.7 s, total: 25.3 s
Wall time: 25.2 s


In [8]:
# Number of rows for each analysis layer 
df_outputs_dropped["analysis_layer"].value_counts()

analysis_layer
net_flux__all_C_pools__all_gases__MgCO2e              2810429
gross_removals__all_C_pools__MgCO2                    1703897
gross_emissions__all_C_pools__all_gases__MgCO2e       1660151
gross_emissions__all_C_pools__CO2_only__MgCO2         1577601
gross_emissions__all_C_pools__non_CO2_only__MgCO2e     426110
Name: count, dtype: int64

In [9]:
# Drops various contextual layers to reduce df size
print(f"Columns before dropping: {df_outputs_dropped.columns}")
# Comment out the contextual layers to keep. 
# Layers on the same line are redundant with each other; they need to be dropped or retained together for full effect.
contextual_layers_to_drop = [   
                             'tile_id', 
                             # cn.adm0_pattern, 'country_name', 'region', 
                             cn.WDPA_pattern, 'WDPA_type', 
                             'WDPA_high_protection',
                             # cn.cont_eco_zstats_pattern, 'continent_ecozone', 'continent', 
                             cn.landmark_pattern, 
                             # cn.starting_composite_primary_forest_pattern,
                             # cn.land_state_pattern, 'land_state_meaning', 
                             # 'land_state_broad_class', 
                             # 'land_state_detailed_class'
                            ]
df_outputs_context_dropped = df_outputs_dropped.drop(columns=contextual_layers_to_drop)

base = len(df_outputs_context_dropped)
print(f"Rows in df with outputs removed, with all contextual layers: {base}")

# Drops specified contextual layers to reduce the number of rows in the table (although dropping adm0 or land_state seems to remove only a few 10s of thousands of rows)
cols_to_sum = ["value", "area_ha", "density__Mg_ha"]

group_cols = [
    c for c in df_outputs_context_dropped.columns
    if c not in contextual_layers_to_drop + cols_to_sum  # Contextual columns to drop
]
# print("group_cols:", group_cols)

df_outputs_context_dropped_agg = (
    df_outputs_context_dropped.groupby(group_cols, dropna=False)
      .sum(numeric_only=True)
      .reset_index()
)
reduced=len(df_outputs_context_dropped_agg)

print(f"Columns after dropping: {df_outputs_context_dropped_agg.columns}")
print(f"Removing tile_id reduces rows by {base - reduced}")
print(f"Rows in output with contextual layers dropped: {reduced}")

Columns before dropping: Index(['analysis_layer', 'adm0', 'land_state_node', 'WDPA', 'cont_eco',
       'Landmark', 'starting_composite_primary_forest', 'year', 'value',
       'tile_id', 'area_ha', 'land_state_meaning', 'land_state_broad_class',
       'land_state_detailed_class', 'country_name', 'region', 'continent',
       'continent_ecozone', 'WDPA_type', 'density__Mg_ha',
       'WDPA_high_protection'],
      dtype='object')
Rows in df with outputs removed, with all contextual layers: 8178188
Columns after dropping: Index(['analysis_layer', 'adm0', 'land_state_node', 'cont_eco',
       'starting_composite_primary_forest', 'year', 'land_state_meaning',
       'land_state_broad_class', 'land_state_detailed_class', 'country_name',
       'region', 'continent', 'continent_ecozone', 'value', 'area_ha',
       'density__Mg_ha'],
      dtype='object')
Removing tile_id reduces rows by 6888717
Rows in output with contextual layers dropped: 1289471


In [10]:
%%time

# Test sums for original and simplified tables. Values should be identical.
year = 2024
variable = cn.net_flux_all_C_pools_all_gases_pattern
print(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())
# print(df_wide_by_land_state[(df_wide_by_land_state['year'] == year)][variable].sum())
print(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())

year = 2020
variable = cn.gross_removals_all_C_pools_pattern
print(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())
# print(df_wide_by_land_state[(df_wide_by_land_state['year'] == year)][variable].sum())
print(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())

year = 2018
variable = cn.gross_emis_all_C_pools_CO2_only_pattern
print(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())
# print(df_wide_by_land_state[(df_wide_by_land_state['year'] == year)][variable].sum())
print(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())

-7278850000.0
-7278849000.0
-18434955000.0
-18434953000.0
10011251000.0
10011249000.0
CPU times: user 2.84 s, sys: 80.6 ms, total: 2.92 s
Wall time: 2.97 s


In [ ]:
# # Way too large to work in PyGWalker (28 million rows)
# walker = pyg.walk(df)

In [ ]:
# # Way too large to work in PyGWalker (8.2 million rows)
# walker = pyg.walk(df_outputs_dropped)

In [12]:
# Loads in PyGWalker (1289471 rows works. NOTE: 1.8 million rows worked for a bit and then crashed)
walker = pyg.walk(df_outputs_context_dropped_agg)

Box(children=(HTML(value='<div id="ifr-pyg-1" style="height: auto">\n    <head>\n        <meta http-equiv="Con…

In [ ]:
# other_full_df = pd.read_parquet(f'{veg_model_parquet_folder}{parquet_core_name}other_outputs_1x1.parquet')
# primary_2015_df = other_full_df[other_full_df['layer_name'] == 'composite_primary_forest_2015']
# primary_2015_df.head()
# primary_2015_df.to_csv(f'/mnt/c/GIS/primary_2015.csv', index=False)

In [ ]:
# # Exports to a csv so data can be used in Excel or reused
# gross_net_aggreg_df.to_csv(f'/mnt/c/GIS/global_iso_aggreg_v{model_version}.csv', index=False)

# gross_net_full_wide_df = gross_net_full_df.pivot(
#     index=['chunk_id', 'years', 'iso'],
#     columns='pattern',
#     values='sum_value'
# ).reset_index()
# gross_net_full_wide_df
# gross_net_full_wide_df.to_csv(f'/mnt/c/GIS/global_chunk_v_{model_version}_wide.csv', index=False)

In [ ]:
# # Groups and sums outputs by country 
# gross_net_aggreg_df = gross_net_full_df.groupby(['pattern', 'years', 'iso', 'tropical'], as_index=False)['sum_value'].sum()
# # gross_net_aggreg_df

In [ ]:
# gross_net_df_example_ISO = gross_net_full_df[gross_net_full_df['iso'] == 'RUS']
# # gross_net_df_RUS

In [ ]:
# gross_net_df_example_chunk = gross_net_full_df[gross_net_full_df['chunk_id'] == '124_-25_125_-24']

In [ ]:
# pyg.walk(gross_net_df_example_ISO, spec=vis_spec)